# Real Estate 10k Dataset

Dataset download (2.5GB): https://google.github.io/realestate10k/ \
Google drive upload is a little slow and finiky...


## Test loading and visualize

In [1]:
from google.colab import drive
# drive.mount('/content/drive')
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import requests
from io import BytesIO

In [3]:
def parse_txt_file(path, max_frames=5):
    with open(path, 'r') as f:
        lines = f.readlines()
    print("read lines")

    video_url = lines[0].strip()
    frames = []

    for line in lines[1:max_frames+1]:
        vals = list(map(float, line.strip().split()))

        timestamp = int(vals[0])

        # Intrinsics (normalized)
        fx, fy, cx, cy = vals[1:5]

        K = np.array([
            [fx, 0, cx],
            [0, fy, cy],
            [0,  0,  1]
        ])

        # Pose (3x4)
        P = np.array(vals[5:17]).reshape(3, 4)

        frames.append({
            "timestamp": timestamp,
            "K": K,
            "P": P
        })

    return video_url, frames

In [4]:
!pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 61.0 MB/s eta 0:00:00


In [5]:
# def load_video_capture(url):
#     try:
#         cap = cv2.VideoCapture(url)
#         if not cap.isOpened():
#             print("Failed to open video:", url) # FAILED HERE
#             return None
#         return cap
#     except:
#         print("Error loading video")
#         return None


import yt_dlp

def get_stream_url(youtube_url):
    ydl_opts = {
        "format": "best[ext=mp4]/best",
        "quiet": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)
        return info["url"]


def load_video_capture(url):
    try:
        stream_url = get_stream_url(url)
        cap = cv2.VideoCapture(stream_url)

        if not cap.isOpened():
            print("Failed to open stream:", stream_url)
            return None

        return cap

    except Exception as e:
        print("Error loading video:", e)
        return None

In [6]:
def show_frames(video_url, frames, num_frames=5):
    cap = load_video_capture(video_url)

    if cap is None:
        print("Skipping video display.")
        return

    for i, frame_data in enumerate(frames[:num_frames]):
        timestamp = frame_data["timestamp"]

        # Convert microseconds → seconds
        time_sec = timestamp / 1e6

        cap.set(cv2.CAP_PROP_POS_MSEC, time_sec * 1000)
        ret, frame = cap.read()

        if not ret:
            print(f"Could not read frame at {time_sec}s")
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(5,5))
        plt.imshow(frame_rgb)
        plt.title(f"Frame {i} @ {time_sec:.2f}s")
        plt.axis('off')
        plt.show()

        print("K (intrinsics):\n", frame_data["K"])
        print("P (pose [R|t]):\n", frame_data["P"])
        print("-" * 50)

    cap.release()

In [7]:
train_dir = "/content/drive/MyDrive/CV Project/dataset/RealEstate10K"
os.listdir(train_dir)

['train', 'test']

In [12]:
train_dir = "/content/drive/MyDrive/CV Project/dataset/RealEstate10K/train/"
test_dir = "/content/drive/MyDrive/CV Project/dataset/RealEstate10K/test/"

sample_path = None

for root, _, files in os.walk(test_dir):
    print(f"test length: {len(files)}")
    # for f in files:
    #     sample_path = os.path.join(root, f)
    #     break

# root = /content/drive/MyDrive/CV Project/dataset/RealEstate10K/train/
for root, _, files in os.walk(train_dir):  # can take a little time, > 70k files
    print(f"train length: {len(files)}")
    for f in files:
        sample_path = os.path.join(root, f)
        break


print(sample_path)

test length: 7711
train length: 71556
/content/drive/MyDrive/CV Project/dataset/RealEstate10K/train/fc7fbf683160c6ac.txt


In [13]:
print(sample_path)
video_url, frames = parse_txt_file(sample_path, max_frames=5) # takes a while...
print("Video URL:", video_url)
show_frames(video_url, frames)

/content/drive/MyDrive/CV Project/dataset/RealEstate10K/train/fc7fbf683160c6ac.txt
read lines
Video URL: https://www.youtube.com/watch?v=S-B3hc2Jsfg


ERROR: [youtube] S-B3hc2Jsfg: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


Error loading video: ERROR: [youtube] S-B3hc2Jsfg: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
Skipping video display.


## Camera pose diversity measurements

In [23]:
import numpy as np


def classify_camera_motion(txt_path, *,
                           frame_range=None,
                           static_translation_eps=1e-4,
                           static_rotation_eps_rad=1e-4,
                           zoom_relative_eps=1e-3,
                           arc_curvature_threshold=0.15,
                           handheld_jerk_threshold=0.5,
                           tracking_alignment_threshold=0.7):
    """Classify camera motion from a RealEstate10K .txt file.

    Args:
        txt_path: path to a RealEstate10K .txt file. First line is the
            YouTube URL; subsequent lines have 19 whitespace-separated
            fields (timestamp, fx, fy, cx, cy, 0, 0, then 12 numbers
            forming the world-to-camera 3x4 [R | t] in row-major order).
        frame_range: optional (start, end) tuple to analyze only a slice
            of frames. Defaults to all frames in the clip.

    Returns:
        dict with 12 keys, each a float:

        Instantaneous shares (sum to 1.0, averaged across consecutive
        frame pairs):
            pan, tilt, roll       - rotations around camera Y / X / Z
            dolly, truck, pedestal - translations along camera Z / X / Y
            zoom                  - relative focal-length change
            static                - residual "no motion" share

        Trajectory-level descriptors (independent, each in [0, 1]):
            arc       - path curvature (orbit-like motion)
            crane     - sustained vertical translation
            tracking  - lateral translation correlated with compensating
                        pan rotation (PROXY for tracking shot)
            handheld  - high-frequency jitter (PROXY for handheld feel)

    Notes:
        * The 8 instantaneous shares need only 2 frames. The 4
          trajectory-level scores need 3+ frames; 5+ recommended.
        * RealEstate10K poses are scale-ambiguous (SfM-recovered).
          Percentages are meaningful; absolute speeds across clips are
          not.
    """
    # -------- read file --------
    with open(txt_path) as f:
        all_lines = [l for l in f.read().splitlines() if l.strip()]
    pose_lines = all_lines[1:]  # drop YouTube URL line
    print(f"First line: {pose_lines[0]}")
    print(len(pose_lines))

    if frame_range is not None:
        start, end = frame_range
        pose_lines = pose_lines[start:end]

    if len(pose_lines) < 2:
        raise ValueError(f"Need at least 2 frame lines, got {len(pose_lines)}.")

    # -------- parse --------
    def _parse(line):
        parts = line.strip().split()
        if len(parts) != 19:
            raise ValueError(f"Expected 19 fields, got {len(parts)}")
        ts = int(parts[0])
        fx, fy, cx, cy = map(float, parts[1:5])
        rt = np.array(list(map(float, parts[7:19])),
                      dtype=np.float64).reshape(3, 4)
        E = np.eye(4)
        E[:3, :4] = rt
        return ts, fx, fy, cx, cy, E

    parsed = [_parse(l) for l in pose_lines]
    extrinsics = [p[5] for p in parsed]
    focals = [0.5 * (p[1] + p[2]) for p in parsed]

    # -------- helpers --------
    def _axis_angle(R):
        cos_theta = np.clip((np.trace(R) - 1.0) / 2.0, -1.0, 1.0)
        theta = np.arccos(cos_theta)
        if theta < 1e-9:
            return np.array([0.0, 0.0, 1.0]), 0.0
        if np.pi - theta < 1e-6:
            M = R + np.eye(3)
            col = M[:, np.argmax(np.linalg.norm(M, axis=0))]
            return col / (np.linalg.norm(col) + 1e-12), theta
        axis = np.array([R[2, 1] - R[1, 2],
                         R[0, 2] - R[2, 0],
                         R[1, 0] - R[0, 1]]) / (2.0 * np.sin(theta))
        return axis, theta

    def _relative(E1, E2):
        rel = E2 @ np.linalg.inv(E1)
        return rel[:3, :3], rel[:3, 3]

    # -------- pairwise instantaneous decomposition --------
    pair_results = []
    for i in range(len(extrinsics) - 1):
        R_rel, t_rel = _relative(extrinsics[i], extrinsics[i + 1])
        axis, theta = _axis_angle(R_rel)

        rot_w = axis ** 2
        tilt_m = rot_w[0] * theta
        pan_m  = rot_w[1] * theta
        roll_m = rot_w[2] * theta

        trans_m = float(np.sum(np.abs(t_rel)))
        truck_m, pedestal_m, dolly_m = (abs(t_rel[0]),
                                        abs(t_rel[1]),
                                        abs(t_rel[2]))

        f1, f2 = focals[i], focals[i + 1]
        zoom_m = abs(f2 - f1) / max(f1, 1e-9)

        if (trans_m < static_translation_eps and
            theta < static_rotation_eps_rad and
            zoom_m < zoom_relative_eps):
            pair_results.append({
                "pan": 0.0, "tilt": 0.0, "roll": 0.0,
                "dolly": 0.0, "truck": 0.0, "pedestal": 0.0,
                "zoom": 0.0, "static": 1.0,
            })
            continue

        total = theta + trans_m + zoom_m
        if total < 1e-12:
            pair_results.append({
                "pan": 0.0, "tilt": 0.0, "roll": 0.0,
                "dolly": 0.0, "truck": 0.0, "pedestal": 0.0,
                "zoom": 0.0, "static": 1.0,
            })
            continue

        pair_results.append({
            "pan":      pan_m / total,
            "tilt":     tilt_m / total,
            "roll":     roll_m / total,
            "dolly":    dolly_m / total,
            "truck":    truck_m / total,
            "pedestal": pedestal_m / total,
            "zoom":     zoom_m / total,
            "static":   0.0,
        })

    averaged = {k: float(np.mean([r[k] for r in pair_results]))
                for k in pair_results[0]}

    # -------- trajectory-level scores --------
    centers = []
    for E in extrinsics:
        R, t = E[:3, :3], E[:3, 3]
        centers.append(-R.T @ t)
    centers = np.stack(centers)

    # Arc: path-to-chord ratio
    arc_score = 0.0
    if len(centers) >= 3:
        seg = np.linalg.norm(np.diff(centers, axis=0), axis=1)
        path_len = float(np.sum(seg))
        chord_len = float(np.linalg.norm(centers[-1] - centers[0]))
        if chord_len > 1e-6 and path_len > 1e-6:
            arc_score = float(np.clip(
                ((path_len / chord_len) - 1.0) / arc_curvature_threshold,
                0.0, 1.0))

    # Crane: vertical translation share
    total_t, vert_t = 0.0, 0.0
    for i in range(len(extrinsics) - 1):
        _, t_rel = _relative(extrinsics[i], extrinsics[i + 1])
        total_t += float(np.linalg.norm(t_rel))
        vert_t += abs(float(t_rel[1]))
    crane_score = vert_t / total_t if total_t > 1e-9 else 0.0

    # Handheld: jerk relative to velocity
    handheld_score = 0.0
    if len(centers) >= 3:
        velocities = np.diff(centers, axis=0)
        accels = np.diff(velocities, axis=0)
        vel_mag = np.linalg.norm(velocities, axis=1).mean() + 1e-9
        acc_mag = np.linalg.norm(accels, axis=1).mean()
        handheld_score = float(np.clip(
            (acc_mag / vel_mag) / handheld_jerk_threshold, 0.0, 1.0))

    # Tracking: anti-correlation of truck and pan
    tracking_score = 0.0
    if len(extrinsics) >= 3:
        x_t, y_r = [], []
        for i in range(len(extrinsics) - 1):
            R_rel, t_rel = _relative(extrinsics[i], extrinsics[i + 1])
            axis, theta = _axis_angle(R_rel)
            x_t.append(t_rel[0])
            y_r.append(axis[1] * theta)
        x_t, y_r = np.array(x_t), np.array(y_r)
        if x_t.std() > 1e-6 and y_r.std() > 1e-6:
            corr = float(-np.corrcoef(x_t, y_r)[0, 1])
            tracking_score = float(np.clip(
                (corr - tracking_alignment_threshold) /
                (1.0 - tracking_alignment_threshold + 1e-9),
                0.0, 1.0))

    return {
        **averaged,
        "arc": arc_score,
        "crane": crane_score,
        "tracking": tracking_score,
        "handheld": handheld_score,
    }


# -------- example usage --------
# result = classify_camera_motion("0a3b5fb184936a83.txt")
# for k, v in result.items():
#     print(f"{k:10s} {v:.3f}")
#
# # or analyze just a window of frames:
# # result = classify_camera_motion("0a3b5fb184936a83.txt", frame_range=(0, 30))

In [24]:
classify_camera_motion(sample_path)

First line: 154821333 0.539167371 0.958519786 0.500000000 0.500000000 0.000000000 0.000000000 0.995712817 0.014784748 0.091309503 0.164167733 -0.014077050 0.999865711 -0.008389745 -0.033649515 -0.091421276 0.007068408 0.995787203 0.046135894
98


{'pan': 0.5924028923505463,
 'tilt': 0.008887902358429097,
 'roll': 0.0010440452198517166,
 'dolly': 0.029495774823199258,
 'truck': 0.3119269672089919,
 'pedestal': 0.05624969755105286,
 'zoom': 0.0,
 'static': 0.0,
 'arc': 0.32038221706060044,
 'crane': 0.178375938642477,
 'tracking': 0.0,
 'handheld': 0.418331982977861}

##